In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from imblearn.over_sampling import SMOTE
from collections import Counter
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Create processed data folder
os.makedirs('../data/processed', exist_ok=True)

print("✅ Libraries loaded!")

✅ Libraries loaded!


In [2]:
df = pd.read_csv('../data/creditcard.csv')
print(f"Loaded dataset: {df.shape}")
print(f"Fraud cases: {df['Class'].sum()}")
df.head()

Loaded dataset: (284807, 31)
Fraud cases: 492


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
# X = all features except Class (the thing we want to predict)
# y = Class column (0 = legit, 1 = fraud)

X = df.drop('Class', axis=1)
y = df['Class']

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape:   {y.shape}")
print(f"\nFeature names: {list(X.columns)}")

Features (X) shape: (284807, 30)
Target (y) shape:   (284807,)

Feature names: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']


In [4]:
# V1 through V28 are already scaled (via PCA in the original dataset).
# But Amount and Time are raw — need scaling.
# We use RobustScaler (handles outliers better than StandardScaler for money data).

scaler = RobustScaler()
X['Amount_scaled'] = scaler.fit_transform(X[['Amount']])
X['Time_scaled'] = scaler.fit_transform(X[['Time']])

# Drop original unscaled versions
X = X.drop(['Amount', 'Time'], axis=1)

print("✅ Amount and Time scaled with RobustScaler")
print(f"\nAmount_scaled stats:\n{X['Amount_scaled'].describe().round(3)}")

# Save the scaler — we'll need it at prediction time to transform new transactions
joblib.dump(scaler, '../models/scaler.pkl')
print("\n💾 Scaler saved to models/scaler.pkl")

✅ Amount and Time scaled with RobustScaler

Amount_scaled stats:
count    284807.000
mean          0.927
std           3.495
min          -0.307
25%          -0.229
50%           0.000
75%           0.771
max         358.683
Name: Amount_scaled, dtype: float64

💾 Scaler saved to models/scaler.pkl


In [5]:
# stratify=y keeps the fraud ratio (0.172%) identical in both train and test.
# Without stratification, random split might put all fraud in train or test — disaster.

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,      # 20% for testing
    random_state=42,     # reproducibility — same split every time
    stratify=y           # ✨ CRITICAL: keeps class ratio intact
)

print("📊 SPLIT RESULTS")
print("=" * 50)
print(f"Training set:   {X_train.shape[0]:,} transactions")
print(f"  - Legit: {(y_train == 0).sum():,}")
print(f"  - Fraud: {(y_train == 1).sum():,} ({y_train.mean()*100:.3f}%)")
print(f"\nTest set:       {X_test.shape[0]:,} transactions")
print(f"  - Legit: {(y_test == 0).sum():,}")
print(f"  - Fraud: {(y_test == 1).sum():,} ({y_test.mean()*100:.3f}%)")
print("\n✅ Fraud % is identical in both → stratification worked")

📊 SPLIT RESULTS
Training set:   227,845 transactions
  - Legit: 227,451
  - Fraud: 394 (0.173%)

Test set:       56,962 transactions
  - Legit: 56,864
  - Fraud: 98 (0.172%)

✅ Fraud % is identical in both → stratification worked


In [7]:
# SMOTE = Synthetic Minority Oversampling Technique
# It creates new synthetic fraud examples by interpolating between real fraud cases.
# Result: training set becomes 50/50 balanced → model actually learns fraud patterns.

# ⚠️ CRITICAL: SMOTE goes on TRAINING data ONLY, never on test data!
# Test data must stay realistic to measure real performance.

print("BEFORE SMOTE:")
print(f"  Legit: {(y_train == 0).sum():,}")
print(f"  Fraud: {(y_train == 1).sum():,}")
print(f"  Ratio: 1:{(y_train == 0).sum() // (y_train == 1).sum()}")
smote = SMOTE(random_state=42, sampling_strategy=1.0)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print("\nAFTER SMOTE:")
print(f"  Legit: {(y_train_balanced == 0).sum():,}")
print(f"  Fraud: {(y_train_balanced == 1).sum():,}")
print(f"  Ratio: 1:1 ✅")
print(f"\nTotal training rows: {len(X_train_balanced):,} (was {len(X_train):,})")

BEFORE SMOTE:
  Legit: 227,451
  Fraud: 394
  Ratio: 1:577

AFTER SMOTE:
  Legit: 227,451
  Fraud: 227,451
  Ratio: 1:1 ✅

Total training rows: 454,902 (was 227,845)


In [8]:
# Save all splits so Phases 5, 6, 7 can load them instantly without redoing this work.

joblib.dump(X_train_balanced, '../data/processed/X_train.pkl')
joblib.dump(y_train_balanced, '../data/processed/y_train.pkl')
joblib.dump(X_test, '../data/processed/X_test.pkl')
joblib.dump(y_test, '../data/processed/y_test.pkl')

# Also save a version of X_train WITHOUT SMOTE (Neural Net sometimes trains better with class_weight)
joblib.dump(X_train, '../data/processed/X_train_original.pkl')
joblib.dump(y_train, '../data/processed/y_train_original.pkl')

print("💾 All processed data saved to data/processed/")
print("\nFiles:")
for f in os.listdir('../data/processed'):
    size = os.path.getsize(f'../data/processed/{f}') / 1024**2
    print(f"  • {f}  ({size:.1f} MB)")

💾 All processed data saved to data/processed/

Files:
  • X_test.pkl  (13.5 MB)
  • X_train.pkl  (104.1 MB)
  • X_train_original.pkl  (53.9 MB)
  • y_test.pkl  (1.7 MB)
  • y_train.pkl  (6.9 MB)
  • y_train_original.pkl  (7.0 MB)


In [9]:
# Quick sanity check that everything loads back correctly
X_train_loaded = joblib.load('../data/processed/X_train.pkl')
y_train_loaded = joblib.load('../data/processed/y_train.pkl')
X_test_loaded = joblib.load('../data/processed/X_test.pkl')
y_test_loaded = joblib.load('../data/processed/y_test.pkl')

print("🔍 SANITY CHECK")
print("=" * 50)
print(f"X_train shape: {X_train_loaded.shape}")
print(f"y_train class counts: {Counter(y_train_loaded)}")
print(f"\nX_test shape: {X_test_loaded.shape}")
print(f"y_test class counts: {Counter(y_test_loaded)}")
print(f"\nFeatures: {list(X_train_loaded.columns)}")
print("\n✅ Everything saved and reloadable. PHASE 4 COMPLETE!")
print("🚀 Ready for Phase 5: Random Forest Training")

🔍 SANITY CHECK
X_train shape: (454902, 30)
y_train class counts: Counter({0: 227451, 1: 227451})

X_test shape: (56962, 30)
y_test class counts: Counter({0: 56864, 1: 98})

Features: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount_scaled', 'Time_scaled']

✅ Everything saved and reloadable. PHASE 4 COMPLETE!
🚀 Ready for Phase 5: Random Forest Training
